# Electrical stimulation fMRI (EPI and SORDINO): analysis

*Version 2.4 of the electrical stimulation notebooks (history in [CHANGELOG.md](CHANGELOG.md)). One notebook for EPI and SORDINO and for both analyses; it replaces `mol_fMRI_electric_stim_analysis.ipynb`, `mol_fMRI_electric_stim_poststim_analysis.ipynb`, `mol_fMRI_sordino_analysis.ipynb` and `mol_fMRI_sordino_poststim_analysis.ipynb` of version 2.3, which are in `archive/v2.3/`.*

Continues on the output of `mol_fMRI_electric_stim_preprocessing_v2.4.ipynb` (or of the v2.3 EPI / SORDINO preprocessing notebooks). Pick the analysis folder of a finished preprocessing run (the folder with `session.json`, e.g. `.../stitched_36-31-40_functionalEPI/analysis`). Folders from older runs without `session.json` are reconstructed from the folder layout. Nothing is recomputed in the preprocessing, so this notebook can be started at any time. The sequence (EPI or SORDINO) is taken from `session.json` (older sessions: from the method file).

1. Load the session and the motion-corrected stitched series.
2. Register to the atlas (once, shared by both parts; an existing registration is reused).
3. **Stimulation response** (optional): the acute response to the stimulation epochs in every atlas region, and voxelwise maps in atlas space.
4. **Post-stimulation (LTP)** (optional): the slow signal change after the stimulation (baseline window vs. signal window) in every atlas region, and voxelwise maps in atlas space.

Tick the parts to run in the first code cell. Each part can be run on its own; both use the same registration and write to the same places as the separate notebooks did (`roi_atlas_<species>/` for the stimulation response, `roi_atlas_<species>/poststim/` for the post-stimulation analysis).

In [ ]:
import electric_stim_helpers, electric_stim_session, electric_stim_atlas, sordino_helpers
from electric_stim_helpers import *
from electric_stim_session import *
from electric_stim_atlas import *
from sordino_helpers import *
from fmri_paths import get_fmri_root

# Version of the stimulation notebooks (electric_stim_session.py); sessions from the preprocessing notebook carry it in session.json
NOTEBOOK_VERSION = electric_stim_session.STIM_NOTEBOOKS_VERSION

# Data root: environment variable FMRI_ROOT, fmri_root.txt or the folder containing this repository (see fmri_paths.py)
FMRI_ROOT = get_fmri_root()

selected_session = FileChooser(f"{FMRI_ROOT}/AnalysedData", layout=widgets.Layout(width="1080px"))
selected_session.show_only_dirs = True
run_stim_entered = widgets.Checkbox(value=True, description="Stimulation response", indent=False)
run_poststim_entered = widgets.Checkbox(value=True, description="Post-stimulation (LTP)", indent=False)
display(selected_session, widgets.HBox([widgets.HTML("<b>Parts to run:</b>&nbsp;"), run_stim_entered, run_poststim_entered]))

print(f"Electrical stimulation notebooks, version {NOTEBOOK_VERSION} (EPI + SORDINO analysis; history: CHANGELOG.md)")

In [ ]:
RUN_STIM = bool(run_stim_entered.value)
RUN_POSTSTIM = bool(run_poststim_entered.value)
if not (RUN_STIM or RUN_POSTSTIM):
    raise ValueError("Tick at least one part (Stimulation response, Post-stimulation) in the cell above.")

session, session_source = load_session(selected_session.selected_path)

in_path = session["in_path"]
subject_id = session["subject_id"]
func_scan_numbers = session["func_scan_numbers"]
struct_scan_number = session["struct_scan_number"]
analysed_func_dir = session["analysed_func_dir"]
analysed_struct_dir = session["analysed_struct_dir"]
analysis_dir = session["analysis_dir"]
mask_file = session["mask_file"]
structural_file_for_coregistration = session["structural_file_for_coregistration"]
scan_ranges = read_scan_ranges(analysed_func_dir)
# Figures: <subject>/figures/QC/<functional folder>_<analysis>/ and figures/results/<functional folder>_<analysis>/ (same folders as the
# preprocessing of this run); the tables, time courses and maps stay in the analysis folder
fig_dirs = figure_folders(analysed_func_dir, analysis_dir, analysed_struct_dir)

os.chdir(analysis_dir)
require_files([analysed_func_dir / "mc_func.nii.gz", analysis_dir / "mean_mc_func.nii.gz", mask_file, structural_file_for_coregistration,
               analysed_func_dir / "scan_boundaries.csv"],
              "Run the preprocessing notebook up to the structural processing first.")

if session_source != str(analysis_dir / SESSION_FILE):
    save_session(analysis_dir, **session)  # next time this folder loads directly

# Sequence: saved by the v2.4 preprocessing; older sessions: SORDINO sessions have scan_info, otherwise the method file decides
sequence_type = session.get("sequence_type") or ("sordino" if session.get("scan_info") else
                                                 detect_sequence_type(Path(in_path) / func_scan_numbers[0]) or "epi")
is_sordino = sequence_type == "sordino"

print_statement(f"Session {session_source}", bcolors.NOTIFICATION)
if session.get("stim_notebooks_version") != NOTEBOOK_VERSION:
    print_statement(f"Note: this session was preprocessed with version {session.get('stim_notebooks_version')} of the stimulation notebooks, "
                    f"this notebook is version {NOTEBOOK_VERSION} (see CHANGELOG.md).", bcolors.NOTIFICATION)
print_statement(f"Subject {subject_id}, {'SORDINO' if is_sordino else 'EPI'}, functional scans {', '.join(func_scan_numbers)} (stitched), structural scan {struct_scan_number}", bcolors.NOTIFICATION)
if session.get("mask_sources"):
    print_statement(f"Brain masks: {session['mask_sources']}", bcolors.NOTIFICATION)
print_statement(f"Analysis folder: {analysis_dir}", bcolors.NOTIFICATION)
for scan, start, end in scan_ranges:
    print(f"  Scan {scan}: volumes {start}-{end}")

# Volume TR: written by the preprocessing (session.json); older sessions read it from the method file of the last scan
if session.get("tr"):
    tr = float(session["tr"])
elif is_sordino:
    tr = float(sordino_param_extract(Path(in_path) / func_scan_numbers[-1])["VolTR"])
else:
    tr = float(func_param_extract(Path(in_path) / func_scan_numbers[-1], export_env=False)["VolTR"])

func_4d = nib.load(str(analysed_func_dir / "mc_func.nii.gz")).get_fdata(dtype=np.float32)
brain = nib.load(mask_file).get_fdata() > 0
if brain.shape != func_4d.shape[:3]:
    raise ValueError(f"Brain mask {brain.shape} and functional data {func_4d.shape[:3]} are on different grids.")
n_total = func_4d.shape[-1]
brain_ts = func_4d[brain].mean(axis=0)
print_statement(f"{n_total} volumes, TR {tr} s, {int(brain.sum())} voxels in the brain mask", bcolors.NOTIFICATION)

# Atlas Coregistration

Select the species in the widget cell below; it fills in the matching atlas:
- **Rat:** SIGMA in vivo template, rigid registration.
- **Mouse:** Allen CCFv3 regions. The structural scan is registered to the DevCCF P56 T2-weighted MRI template (ex vivo, 50 µm; Kronman et al. 2024), which contains the Allen CCFv3 template and annotation warped into its space. The template is brain-masked, downsampled to 0.1 mm for registration, and registered with a scaling term and `nmi` cost. Regions: the 316 Allen summary structures (cortical layers merged), split into left and right (`_L` / `_R`). The files are made once with `tools/build_allen_atlas.py` (in `atlas/Allen_DevCCF_P56/`). Until v2.2 the mouse atlas was DSURQE; region names and ids differ, so do not mix older and newer mouse results.

Follows `roi_atlas_analysis.py`: the cleaned structural scan is registered to the brain-masked functional mean (struct2func, rigid, bounded to 5° and 2 mm like `mol_fMRI_v5.6`) and to the atlas template (struct2atlas), and the two matrices are composed into one functional → atlas matrix (`register_to_atlas` in `electric_stim_atlas.py`). The multi-label atlas is resampled once onto the functional grid with nearest-neighbour interpolation, so the functional data itself is never interpolated. Outputs go to `roi_atlas_<species>/`; a registration that already exists is reused (also one made by the v2.3 analysis notebooks). A quality control of the three registration results (structural → functional, structural → atlas, atlas labels on the functional grid) follows: overlay images and reports in `roi_atlas_<species>/qc/`, with a warning for anything that looks wrong (a transform at its limit, poor brain mask overlap, little of the brain covered by labels). SORDINO: the mean image has a zero-echo-time contrast and a lower resolution (0.6 mm) than the EPI, so the registration settings are not tested for it; check the quality control images.

Optional label names CSV: either columns `id,name` (the Allen list `allen_summary_labels.csv`), or the SIGMA structure list with separate left and right label columns (regions then get an `_L` / `_R` suffix). Without it, regions are named `label_<id>`.

In [ ]:
SPECIES_ATLAS = species_atlas_config(FMRI_ROOT)

atlas_layout = widgets.Layout(width="700px")
atlas_style = {"description_width": "160px"}
species_dropdown = widgets.Dropdown(options=list(SPECIES_ATLAS), value="rat", description="Species:", layout=widgets.Layout(width="280px"), style=atlas_style)
atlas_template_entered = widgets.Text(description="Atlas template:", layout=atlas_layout, style=atlas_style)
atlas_labels_entered = widgets.Text(description="Atlas labels:", layout=atlas_layout, style=atlas_style)
atlas_mask_entered = widgets.Text(placeholder="optional, restricts the template to the brain", description="Template brain mask:", layout=atlas_layout, style=atlas_style)
atlas_names_entered = widgets.Text(placeholder="optional CSV (id,name or left/right label columns)", description="Label names CSV:", layout=atlas_layout, style=atlas_style)


def fill_atlas_defaults(change=None):
    cfg = SPECIES_ATLAS[species_dropdown.value]
    atlas_template_entered.value = cfg["template"]
    atlas_labels_entered.value = cfg["labels"]
    atlas_mask_entered.value = cfg["mask"]
    atlas_names_entered.value = cfg["names"]


species_dropdown.observe(fill_atlas_defaults, names="value")
fill_atlas_defaults()

display(widgets.VBox([species_dropdown, atlas_template_entered, atlas_labels_entered, atlas_mask_entered, atlas_names_entered],
    layout=widgets.Layout(padding="12px", border="3px solid #444", border_radius="10px", background_color="#1e1e1e")))

In [ ]:
OPEN_FSLEYES = True  # look at the registration / the maps in fsleyes

species = species_dropdown.value
species_cfg = SPECIES_ATLAS[species]
atlas_template = Path(atlas_template_entered.value.strip()).resolve()
atlas_labels = Path(atlas_labels_entered.value.strip()).resolve()
atlas_mask_text = atlas_mask_entered.value.strip()
for f in (atlas_template, atlas_labels) + ((Path(atlas_mask_text),) if atlas_mask_text else ()):
    if not f.is_file():
        raise FileNotFoundError(f"{f} (atlas files for {species} not found; check the paths above)")

atlas_dir = analysis_dir / f"roi_atlas_{species}"
atlas_dir.mkdir(exist_ok=True)
electric_stim_atlas.ensure_atlas_folder(atlas_dir, atlas_template, atlas_labels)  # a folder made with another atlas is set aside
func_reference = analysis_dir / "mean_mc_func.nii.gz"

print_header(f"Registering structural to functional and to the {species} atlas", bcolors.HEADER)
composite, labels_funcspace = register_to_atlas(Path(structural_file_for_coregistration), func_reference, mask_file, atlas_dir, species_cfg,
                                                atlas_template, atlas_labels, atlas_mask_text)

# Quality control of the registrations (overlay images in the QC figure folder, reports in the qc folder, warnings are printed)
registration_qc(structural_file_for_coregistration, func_reference, mask_file, atlas_dir, species_cfg, atlas_template, atlas_labels,
                labels_funcspace, struct_mask=structural_mask_file(analysed_struct_dir), atlas_mask=atlas_mask_text,
                figure_dir=fig_dirs["qc"])

# Check the registration visually before trusting the regions
if OPEN_FSLEYES:
    subprocess.run(["fsleyes", str(func_reference), str(labels_funcspace), "-cm", "random", "-a", "40"])

# Atlas regions on the functional grid, inside the brain mask (used by both parts)
labels_img = nib.load(str(labels_funcspace))
labels = np.rint(labels_img.get_fdata()).astype(int)
if brain.shape != labels.shape:
    raise ValueError(f"Brain mask {brain.shape} and atlas labels {labels.shape} are on different grids.")
labels[~brain] = 0
label_names = load_label_names(atlas_names_entered.value.strip()) if atlas_names_entered.value.strip() else {}


def save_map(data, path):
    """Saves a map on the functional grid as float32 (the label image header would otherwise cast it to integers)."""
    header = labels_img.header.copy()
    header.set_data_dtype(np.float32)
    nib.save(nib.Nifti1Image(data.astype(np.float32), labels_img.affine, header), str(path))

# Part 1: Stimulation Response

## Stimulation Setup

Locates the stimulation scan in the stitched series and derives the epochs. The stimulation check itself is run on the mean time course of every atlas region below, so no manually drawn ROIs are needed. Volumes dropped at the start of each scan in the preprocessing (`drop_vols` in `session.json`) are taken into account.

**EPI:** the stimulation scan is the one with `FMRIYesNo=Yes` in its Bruker `method` file. Stimulation: 5 Hz pulses (200 ms interval, 100 µs width, 3 mA), 40 s ON / 20 s OFF per 60 s epoch. The epoch timing is read from the `method` file (`PreBaselineNum`, `StimNum` + `InterStimNum`, `NEpochs`); the ON/OFF split is entered below (default 40 / 20 s) because the `method` file only stores the 60 s period; the dropped volumes are subtracted from the pre-stimulation baseline. Percent signal change is relative to the pre-stimulation baseline at the start of the stimulation scan. A paired t-test across epochs compares the ON window (after the hemodynamic lag) with the rest period right before each onset (`OFF window` field, default 10 s).

**SORDINO:** the stimulation scan is the one with block triggers (`BlockTriggerMode=On`). The scanner only sends a trigger pulse at the start of every block (`NTrigBlocks`, `NVolumesPerBlock`) to the Master-9, which runs the train (5 Hz, 5 ms pulses, 3 mA). Each trigger block is one epoch (a trigger block that is at least twice as long as the shortest one is the post-stimulation phase of the same scan and is not an epoch): the train lasts *Stimulation ON* and the rest of the block is OFF. The Master-9 train length cannot be read from the data, so **enter the stimulation ON time (seconds)**; it has to match the Master-9 setting (e.g. 40 s, or 45 s = N = 225 pulses at 5 Hz). The OFF time is the rest of the trigger block unless you enter it. The trigger block set in ParaVision has to be longer than the train (otherwise there is no OFF period and the notebook stops). A block whose trigger lies in the dropped volumes is skipped. Percent signal change is relative to the mean of the OFF windows of the stimulation scan. Every epoch is compared with itself: a paired t-test across epochs compares the ON window (after the hemodynamic lag) with the OFF window at the end of the same block (the last `OFF window` seconds before the next trigger, after the response to the train has decayed). Because both windows are at most 60 s apart, level differences between scans and slow drift cancel out, so no baseline scan is needed.

### Timing fields (input cell below)

All times are in seconds. They are rounded to whole volumes (SORDINO 2.4 s per volume, EPI the TR of the scan), so the windows printed by the setup cell can differ from what you entered by up to half a volume; the setup cell prints the windows that are actually used, in seconds and volumes. One epoch = one stimulation period (60 s: train + rest).

| Field | What it is | What it is used for | Default | Effect if it is changed |
|---|---|---|---|---|
| **Stimulation ON** | Length of the stimulation train after every onset (SORDINO: the Master-9 setting). | End of the ON window; orange bars in the figures. | EPI 40 s; SORDINO required | **Must match the real train.** Too long: the ON window includes the time after the train, when the signal already falls, so the response is underestimated (SORDINO: the notebook stops if the train would leave no OFF period). Too short: the end of the response is left out (fewer volumes, noisier). |
| **Stimulation OFF** | Rest after the train until the next onset. | **Only a check**: ON + OFF has to equal the epoch length (EPI: `method` file; SORDINO: trigger block in ParaVision), otherwise a red warning. Not used in the calculation. | EPI 20 s; SORDINO 0 = the rest of the trigger block | None on the results. |
| **Hemodynamic lag** | The blood (BOLD/CBV) response starts a few seconds after the stimulation and needs some seconds to rise. These first seconds are left out of the ON window. | Start of the ON window = onset + lag. | 5 s | 0 s: the rising part is averaged into the ON window, which lowers the ON signal and the t-values a little. Larger: the ON window gets shorter (noisier), but contains only the plateau. At least 3 volumes have to be left after the lag. |
| **OFF window at block end** (SORDINO) / **Rest window before onset** (EPI) | The baseline every epoch is compared with: the last N seconds before the next onset, when the signal is back at rest. | Paired t-test across epochs, ON window vs. this window, epoch by epoch. SORDINO: also the reference of the percent signal change (mean of all OFF windows), so it changes the `%` values too. EPI: the percent signal change is relative to the pre-stimulation baseline; this window is only the comparison. | 10 s | **Too short** (1–2 volumes): the baseline of every epoch is a single noisy value, so the t-values drop and fewer regions are significant. **Too long**: it reaches back to the end of the train, where the signal has not returned to rest yet (or undershoots), so the ON–OFF difference is wrong; EPI: longer than the OFF phase reaches into the previous train. About half of the OFF phase (10 s of 20 s) is a good compromise. |

Example (SORDINO, 60 s trigger blocks, 2.4 s per volume) with the defaults 40 / 0 / 5 / 10 s: ON window 4.8–40.8 s after the trigger (15 volumes), OFF window 50.4–60 s (4 volumes). With lag 0 and OFF window 0 instead, the ON window is 0–40.8 s (17 volumes) and the OFF window only the last volume (57.6–60 s, the minimum of 1 volume).

In [ ]:
if RUN_STIM:
    # Stimulation timing. SORDINO: not stored in the data (the scanner only sends the trigger, the Master-9 runs the train), so the
    # ON time is required. EPI: the method file stores only the 60 s period, the defaults are the 40 / 20 s split.
    input_layout = widgets.Layout(width="320px", flex="0 0 auto")
    label_style = {"description_width": "230px"}
    stim_on_entered = widgets.FloatText(value=0 if is_sordino else 40, description="Stimulation ON (s, required):" if is_sordino else "Stimulation ON (s):",
                                        layout=input_layout, style=label_style)
    stim_off_entered = widgets.FloatText(value=0 if is_sordino else 20, description="Stimulation OFF (s, 0 = rest of block):" if is_sordino else "Stimulation OFF (s):",
                                         layout=input_layout, style=label_style)
    lag_entered = widgets.FloatText(value=5, description="Hemodynamic lag (s):", layout=input_layout, style=label_style)
    pre_entered = widgets.FloatText(value=10, description="OFF window at block end (s):" if is_sordino else "Rest window before onset (s):",
                                    layout=input_layout, style=label_style)
    display(widgets.HBox([stim_on_entered, stim_off_entered, lag_entered, pre_entered],
        layout=widgets.Layout(width="100%", flex_flow="row wrap", justify_content="space-between", align_items="center", padding="12px", border="3px solid #444", border_radius="10px", background_color="#1e1e1e")))
    display(widgets.HTML(f"<i>Seconds, rounded to whole volumes ({tr:.2f} s per volume). Stimulation OFF is only a check. "
                         "Lag: left out at the start of the ON window. "
                         + ("OFF window: the last seconds of every block, the baseline of each epoch. " if is_sordino else "Rest window: the seconds before every onset, the baseline of each epoch. ")
                         + "Recommended lag 5 s, window 10 s; see the table above for the effect of each field.</i>"))

    # For orientation: the trigger blocks of the scans (SORDINO, read from the method files)
    for num, info in session.get("scan_info", {}).items():
        if info.get("BlockTriggerMode"):
            print_statement(f"Scan {num}: trigger blocks of {[round(n * info['VolTR']) for n in info['volumes_per_block']]} s", bcolors.NOTIFICATION)
else:
    print_statement("⏭️ Stimulation response not selected (tick it in the first cell)", bcolors.NOTIFICATION)

In [ ]:
if RUN_STIM:
    from scipy import stats

    # Timing from the input cell above
    STIM_ON_S = float(stim_on_entered.value)  # length of the stimulation (train) after every onset
    LAG_S = float(lag_entered.value)  # hemodynamic delay excluded from the start of the ON window
    PRE_S = float(pre_entered.value)  # comparison window: EPI the rest right before each onset, SORDINO the end of every block
    if STIM_ON_S <= 0:
        raise ValueError("Enter the length of the stimulation (Stimulation ON, in seconds) in the input cell above and run this cell again.")

    if not scan_ranges:
        raise RuntimeError("scan_boundaries.csv not found, cannot locate the stimulation scan in the stitched series.")
    drop_vols = session.get("drop_vols", 0)

    if is_sordino:
        stim_idx = find_trigger_scan(in_path, func_scan_numbers)
        if stim_idx is None:
            raise ValueError(f"None of the scans {func_scan_numbers} has BlockTriggerMode=On in its method file.")
        stim_num = func_scan_numbers[stim_idx]
        p_stim = sordino_param_extract(Path(in_path) / stim_num)
    else:
        stim_idx = next((i for i, num in enumerate(func_scan_numbers)
                         if re.search(r"##\$FMRIYesNo=Yes", (Path(in_path) / num / "method").read_text())), None)
        if stim_idx is None:
            raise ValueError(f"None of the scans {func_scan_numbers} has FMRIYesNo=Yes in its method file.")
        stim_num = func_scan_numbers[stim_idx]
        p_stim = func_param_extract(Path(in_path) / stim_num, export_env=False)
    stim_tr = p_stim["VolTR"]
    _, stim_start, stim_end = scan_ranges[stim_idx]
    stim_end += 1  # exclusive

    lag_vols = int(round(LAG_S / stim_tr))
    on_vols = int(round(STIM_ON_S / stim_tr))

    if is_sordino:
        # Trigger blocks as volume indices of the stitched series; volumes dropped at the start of every scan shift them
        blocks = [(stim_start + o - drop_vols, n) for o, n in zip(p_stim["block_onsets"], p_stim["volumes_per_block"]) if o >= drop_vols]
        if len(blocks) < len(p_stim["block_onsets"]):
            print_statement(f"{len(p_stim['block_onsets']) - len(blocks)} trigger block(s) lie in the dropped volumes and are skipped.", bcolors.FAIL)
        if len(blocks) < 3:
            raise ValueError(f"Only {len(blocks)} usable trigger blocks in scan {stim_num}; at least 3 are needed for the statistics.")

        # One epoch is one trigger block; all epochs need the same length. A trigger block that is much longer than the others
        # (e.g. 500 volumes after ten blocks of 25 volumes) is the post-stimulation phase of the same scan, not an epoch.
        period_vols = min(n for _, n in blocks)
        for s, n in blocks:
            if n >= 2 * period_vols:
                print_statement(f"Trigger block at volume {s} is {n * stim_tr:.0f} s long: treated as the post-stimulation phase, not as an epoch.", bcolors.NOTIFICATION)
        blocks = [(s, n) for s, n in blocks if n < 2 * period_vols]
        if len({n for _, n in blocks}) > 1:
            print_statement(f"The trigger blocks have different lengths ({sorted({n for _, n in blocks})} volumes); the epochs are cut to the shortest one.", bcolors.FAIL)
        STIM_OFF_S = float(stim_off_entered.value) or period_vols * stim_tr - STIM_ON_S  # 0 = the rest of the trigger block
        if abs(period_vols * stim_tr - (STIM_ON_S + STIM_OFF_S)) > stim_tr:
            print_statement(f"The trigger block ({period_vols * stim_tr:.0f} s) differs from STIM_ON_S + STIM_OFF_S ({STIM_ON_S + STIM_OFF_S} s): check the block length in ParaVision and the Master-9 train.", bcolors.FAIL)

        pre_vols = max(1, int(round(PRE_S / stim_tr)))
        rest_start = period_vols - pre_vols  # OFF window: the last pre_vols volumes of the block
        if on_vols - lag_vols < 3:
            raise ValueError("The ON window after the hemodynamic lag LAG_S is shorter than 3 volumes.")
        if rest_start < on_vols:
            raise ValueError(f"The block ({period_vols * stim_tr:.0f} s) has no OFF period after the {STIM_ON_S} s train (OFF window would start at {rest_start * stim_tr:.0f} s). "
                             "Check STIM_ON_S and the Master-9 train length: the train has to be shorter than the trigger block.")
        onsets = [s for s, _ in blocks if s + period_vols <= stim_end]
        if len(onsets) < 3:
            raise ValueError(f"Only {len(onsets)} complete epochs in scan {stim_num}; at least 3 are needed for the statistics.")
        epoch_offset = 0  # an epoch runs from its trigger to the next one
    else:
        STIM_OFF_S = float(stim_off_entered.value)
        # The first volumes of every scan can be dropped in the preprocessing, which shortens the baseline at the start of the scan
        n_base_vols = p_stim["Baseline_TRs"] - drop_vols
        if n_base_vols <= 0:
            raise ValueError("The dropped volumes cover the whole pre-stimulation baseline of the stimulation scan.")
        period_vols = p_stim["StimOn_TRs"] + p_stim["StimOff_TRs"]
        if abs(period_vols * stim_tr - (STIM_ON_S + STIM_OFF_S)) > 1e-6:
            print_statement(f"Epoch period in method file ({period_vols * stim_tr:.0f} s) differs from STIM_ON_S + STIM_OFF_S.", bcolors.FAIL)
        pre_vols = max(1, int(round(PRE_S / stim_tr)))  # at least one volume
        onsets = [stim_start + n_base_vols + k * period_vols for k in range(p_stim["NoOfEpochs"])]
        onsets = [o for o in onsets if o + period_vols <= stim_end]
        epoch_offset = -pre_vols  # an epoch includes the rest window before its onset


    def window_means(x):
        """Mean of the ON window (after the lag) and of the comparison window of every epoch (EPI: the rest right before the onset,
        SORDINO: the OFF window at the end of the block); last axis of x is time, epochs are stacked on a new last axis."""
        on = np.stack([x[..., o + lag_vols:o + on_vols].mean(axis=-1) for o in onsets], axis=-1)
        if is_sordino:
            rest = np.stack([x[..., o + rest_start:o + period_vols].mean(axis=-1) for o in onsets], axis=-1)
        else:
            rest = np.stack([x[..., o - pre_vols:o].mean(axis=-1) for o in onsets], axis=-1)
        return on, rest


    def baseline_level(x):
        """Reference of the percent signal change (last axis is time). EPI: pre-stimulation baseline at the start of the stimulation scan;
        SORDINO: mean of the OFF windows of the stimulation scan."""
        if is_sordino:
            return window_means(x)[1].mean(axis=-1)
        return x[..., stim_start:stim_start + n_base_vols].mean(axis=-1)


    rest_label = "OFF" if is_sordino else "rest"
    print_statement(f"Stimulation scan {stim_num} (volumes {stim_start}-{stim_end - 1}), TR {stim_tr:.3f} s: {len(onsets)} epochs of {period_vols * stim_tr:.0f} s, first onset at volume {onsets[0]}", bcolors.NOTIFICATION)
    # The windows actually used (rounded to whole volumes), and warnings for settings that weaken the comparison
    comp_s = pre_vols * stim_tr
    if is_sordino:
        comp_text = f"OFF window {rest_start * stim_tr:.1f}-{period_vols * stim_tr:.1f} s after each trigger ({pre_vols} volumes, the end of the block)"
    else:
        comp_text = f"rest window {comp_s:.1f}-0 s before each onset ({pre_vols} volumes)"
    print_statement(f"ON window {lag_vols * stim_tr:.1f}-{on_vols * stim_tr:.1f} s after each onset ({on_vols - lag_vols} volumes, lag {lag_vols * stim_tr:.1f} s); "
                    f"compared with: {comp_text}", bcolors.NOTIFICATION)
    if comp_s < 5:
        print_statement(f"The comparison window is only {comp_s:.1f} s ({pre_vols} volume(s)): the baseline of every epoch is noisy, which lowers the t-values. "
                        "10 s is recommended.", bcolors.FAIL)
    if lag_vols == 0:
        print_statement("No hemodynamic lag: the ON window starts while the signal is still rising, which lowers the ON signal a little. 5 s is recommended.", bcolors.NOTIFICATION)
    if is_sordino and rest_start * stim_tr - STIM_ON_S < 5:
        print_statement(f"The OFF window starts {rest_start * stim_tr - STIM_ON_S:.1f} s after the end of the train: the response may not be back at rest yet, "
                        "which makes the ON-OFF difference too small. Use a shorter OFF window.", bcolors.FAIL)
    if not is_sordino and PRE_S > STIM_OFF_S:
        print_statement(f"The rest window ({PRE_S:.0f} s) is longer than the OFF phase ({STIM_OFF_S:.0f} s): it reaches into the stimulation of the previous epoch. "
                        "Use a shorter rest window.", bcolors.FAIL)
else:
    print_statement("⏭️ Stimulation response not selected (tick it in the first cell)", bcolors.NOTIFICATION)

## Stimulation Response per Brain Region

The mean time course of every atlas region (inside the brain mask) is extracted and the stimulation check (paired t-test across epochs, ON window vs. the comparison window, see above) is run on each region, with Benjamini-Hochberg FDR correction across regions. Requires the stimulation setup cell above to have been run. This replaces manually drawn ROIs. Outputs (in `roi_atlas_<species>/`):
- `atlas_roi_stim_response.csv`: every region with the ON and comparison signal change (`on_psc`, `rest_psc`), their difference (`diff_psc`, %), t, p and FDR-corrected q; `atlas_roi_responders.csv` lists only the regions with a significant positive response.
- Figures (in `<subject>/figures/results/<functional folder>_<analysis>/stim_response/`): `atlas_roi_stim_response_top_increases.svg` / `..._top_decreases.svg`: epoch averages of the `N_TOP` regions with the strongest signal increase and the strongest decrease (ranked by t, or by the size of the signal change with `RANK_BY`); the same regions are printed as tables.
- `time_courses/time_course_roi_<id>_<name>.txt`: mean time course of every region; `atlas_roi_timeseries_all.csv`: all of them.
- `atlas_roi_stim_response_map.nii.gz`: signal change of the responding regions on the functional grid.
- `stim_check_plots/` (in the same figure folder): one figure per region (stimulation scan with the stimulation periods marked, and the epoch average ± SEM); the responders are also in `stim_check_plots/responders/`.
- The somatosensory (S1) regions are printed separately as a check that the stimulation worked.

In [ ]:
if RUN_STIM:
    from statsmodels.stats.multitest import multipletests

    MIN_VOXELS = 5
    FDR_ALPHA = 0.05
    N_TOP = 10  # regions shown per direction: the N_TOP strongest signal increases and the N_TOP strongest decreases
    RANK_BY = "t"  # "t": by t-statistic (significance); "diff_psc": by size of the signal change in % (favours small, noisy regions)
    SAVE_ALL_REGION_PLOTS = True  # one stim-check figure per region in stim_check_plots/ (responders always saved)
    S1_PATTERN = r"somatosens|\bS1"  # regions printed as the stimulation sanity check


    def stim_response(ts):
        base = baseline_level(ts)
        psc = (ts - base) / base * 100
        epochs = np.array([psc[o + epoch_offset:o + period_vols] for o in onsets])
        on, rest = window_means(psc)
        t_stat, p_val = stats.ttest_rel(on, rest)
        return epochs, on.mean(), rest.mean(), t_stat, p_val


    tc_dir = atlas_dir / "time_courses"
    tc_dir.mkdir(exist_ok=True)
    stim_fig_dir = fig_dirs["results"] / "stim_response"  # figures; tables and time courses stay in atlas_dir
    plot_dir = stim_fig_dir / "stim_check_plots"
    plot_dir.mkdir(parents=True, exist_ok=True)
    (plot_dir / "responders").mkdir(exist_ok=True)

    rows, region_ts, region_epochs = [], {}, {}
    for lid in np.unique(labels[labels > 0]):
        sel = labels == lid
        n_vox = int(sel.sum())
        if n_vox < MIN_VOXELS:
            continue
        ts = func_4d[sel].mean(axis=0)
        if baseline_level(ts) <= 0:
            continue
        epochs, on_psc, rest_psc, t_stat, p_val = stim_response(ts)
        np.savetxt(tc_dir / f"time_course_roi_{lid}_{safe_name(label_names.get(int(lid), f'label_{lid}'))}.txt", ts)
        region_ts[lid] = ts
        region_epochs[lid] = epochs
        rows.append({"label_id": int(lid), "name": label_names.get(int(lid), f"label_{lid}"), "n_voxels": n_vox,
                     "on_psc": on_psc, "rest_psc": rest_psc, "diff_psc": on_psc - rest_psc, "t": t_stat, "p": p_val})

    results = pd.DataFrame(rows)
    valid = results["p"].notna()
    results.loc[valid, "q_fdr"] = multipletests(results.loc[valid, "p"], alpha=FDR_ALPHA, method="fdr_bh")[1]
    results = results.sort_values("p").reset_index(drop=True)
    results["responder"] = (results["q_fdr"] < FDR_ALPHA) & (results["diff_psc"] > 0)

    results.to_csv(atlas_dir / "atlas_roi_stim_response.csv", index=False)
    results[results["responder"]].to_csv(atlas_dir / "atlas_roi_responders.csv", index=False)
    pd.DataFrame({results.loc[results.label_id == lid, "name"].iloc[0]: ts for lid, ts in region_ts.items()}).to_csv(
        atlas_dir / "atlas_roi_timeseries_all.csv", index_label="timepoint")

    print_statement(f"{len(results)} regions analysed, {int(results['responder'].sum())} with significant positive response (FDR q < {FDR_ALPHA})", bcolors.OKGREEN)
    print(results.head(20).to_string(index=False))

    s1 = results[results["name"].str.contains(S1_PATTERN, case=False, regex=True)]
    print_statement("Somatosensory (S1) regions:" if len(s1) else "No region name matches S1_PATTERN (provide a label names CSV).", bcolors.NOTIFICATION)
    if len(s1):
        print(s1.to_string(index=False))

    # Map of the ON signal change relative to the comparison window in regions that respond
    response_map = np.zeros(labels.shape, dtype=np.float32)
    for r in results[results["responder"]].itertuples():
        response_map[labels == r.label_id] = r.diff_psc
    save_map(response_map, atlas_dir / "atlas_roi_stim_response_map.nii.gz")

    # Epoch averages of the strongest regions: the N_TOP strongest signal increases and the N_TOP strongest decreases
    t_epoch = np.arange(epoch_offset, period_vols) * stim_tr


    def mark_epoch(ax):
        ax.axvspan(0, STIM_ON_S, color="orange", alpha=0.3)
        if is_sordino:
            ax.axvspan(rest_start * stim_tr, period_vols * stim_tr, color="grey", alpha=0.3)


    def top_regions(sign):
        """The N_TOP regions with the strongest signal increase (sign=1) or decrease (sign=-1), ranked by RANK_BY."""
        column = "t" if RANK_BY == "t" else "diff_psc"
        candidates = results[(results["diff_psc"] * sign > 0) & results[column].notna()]
        return candidates.loc[(candidates[column] * sign).sort_values(ascending=False).index[:N_TOP]]


    def plot_top_regions(top, title, out_file):
        n_rows = max(1, int(np.ceil(len(top) / 4)))
        fig, axes = plt.subplots(n_rows, 4, figsize=(18, 3.5 * n_rows), squeeze=False)
        for ax, r in zip(axes.flat, top.itertuples()):
            ep = region_epochs[r.label_id]
            m = ep.mean(axis=0)
            sem = ep.std(axis=0, ddof=1) / np.sqrt(len(ep))
            ax.plot(t_epoch, m)
            ax.fill_between(t_epoch, m - sem, m + sem, alpha=0.3)
            mark_epoch(ax)
            ax.set_title(f"{r.name}\nΔ={r.diff_psc:.2f}%, q={r.q_fdr:.3g}", fontsize=9)
            ax.set_xlabel("Time from onset (s)")
            ax.set_ylabel("Signal change (%)")
        for ax in list(axes.flat)[len(top):]:
            ax.axis("off")
        fig.suptitle(title)
        fig.tight_layout(rect=(0, 0, 1, 0.97))
        fig.savefig(out_file, dpi=300)
        plt.show()


    top_columns = ["name", "n_voxels", "on_psc", "rest_psc", "diff_psc", "t", "p", "q_fdr"]
    for sign, label in ((1, "increase"), (-1, "decrease")):
        top = top_regions(sign)
        print_statement(f"{len(top)} strongest signal {label}s (ranked by {RANK_BY}):", bcolors.NOTIFICATION)
        print(top[top_columns].to_string(index=False))
        plot_top_regions(top, f"Strongest signal {label}s (ranked by {RANK_BY})", stim_fig_dir / f"atlas_roi_stim_response_top_{label}s.svg")


    def save_region_plot(r):
        ts, ep = region_ts[r.label_id], region_epochs[r.label_id]
        base = baseline_level(ts)
        psc = (ts - base) / base * 100
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))
        ax1.plot((np.arange(stim_start, stim_end) - stim_start) * stim_tr, psc[stim_start:stim_end], lw=0.8)
        for o in onsets:
            ax1.axvspan((o - stim_start) * stim_tr, (o - stim_start) * stim_tr + STIM_ON_S, color="orange", alpha=0.3)
            if is_sordino:
                ax1.axvspan((o + rest_start - stim_start) * stim_tr, (o + period_vols - stim_start) * stim_tr, color="grey", alpha=0.3)
        ax1.set_xlabel("Time in stimulation scan (s)")
        ax1.set_ylabel("Signal change (%)")
        ax1.set_title(f"{r.name} ({r.n_voxels} voxels): stimulation scan {stim_num} (orange = stimulation ON{', grey = OFF window' if is_sordino else ''})")
        m = ep.mean(axis=0)
        sem = ep.std(axis=0, ddof=1) / np.sqrt(len(ep))
        ax2.plot(t_epoch, m)
        ax2.fill_between(t_epoch, m - sem, m + sem, alpha=0.3)
        mark_epoch(ax2)
        ax2.axvline(0, color="k", lw=0.5)
        ax2.set_xlabel("Time from stimulation onset (s)")
        ax2.set_ylabel("Signal change (%)")
        ax2.set_title(f"Epoch average ± SEM (n={len(ep)}), ON {r.on_psc:.2f}% vs {rest_label} {r.rest_psc:.2f}%, "
                      f"t={r.t:.2f}, p={r.p:.3g}, q={r.q_fdr:.3g}")
        fig.tight_layout()
        fname = f"stim_check_{r.label_id}_{safe_name(r.name)}.svg"
        if SAVE_ALL_REGION_PLOTS:
            fig.savefig(plot_dir / fname, dpi=300)
        if r.responder:
            fig.savefig(plot_dir / "responders" / fname, dpi=300)
        plt.close(fig)


    for r in results.itertuples():
        if SAVE_ALL_REGION_PLOTS or r.responder:
            save_region_plot(r)
    print_statement(f"Stim-check figures saved in {plot_dir} ({int(results['responder'].sum())} responders in {plot_dir / 'responders'})", bcolors.OKGREEN)
    print_statement(f"Region time courses saved in {tc_dir}", bcolors.OKGREEN)
else:
    print_statement("⏭️ Stimulation response not selected (tick it in the first cell)", bcolors.NOTIFICATION)

## Voxelwise Stimulation Maps in Atlas Space (for group analysis)

The region analysis above stays in native space, so the functional time series are never interpolated. For voxelwise group statistics, the maps are computed in native space and then moved to atlas space with the inverse of the functional → atlas matrix estimated above (struct2func and struct2atlas), in a single resampling step:
- `stim_diff_psc`: mean ON − comparison-window signal change (%) per voxel, relative to the reference of the percent signal change (EPI: the pre-stimulation baseline; SORDINO: the mean OFF signal).
- `stim_tstat`: paired t of ON vs. the comparison window across epochs per voxel (n epochs − 1 degrees of freedom).
- `mean_func` and `brain_mask`, for overlays and for restricting the group statistics to the brain.

The maps contain signal increases only (negative values are set to 0); set `POSITIVE_ONLY = False` in the code cell to keep decreases as well. Outputs go to `roi_atlas_<species>/atlas_space/` on the registration template grid, which is the same for every animal of the species. Region-wise group comparisons can use `atlas_roi_stim_response.csv` directly, since the atlas labels already define the same regions in every animal.

In [ ]:
if RUN_STIM:
    POSITIVE_ONLY = True  # only signal increases in the maps: negative values are set to 0 (False keeps both directions)

    # Voxelwise maps in native space (same definition as the region analysis: ON window after the lag vs. the comparison window)
    on_means, rest_means = window_means(func_4d)
    base_img = rest_means.mean(axis=-1) if is_sordino else baseline_level(func_4d)
    valid_vox = brain & (base_img > 0)
    diffs = (on_means - rest_means) / np.where(valid_vox, base_img, 1)[..., None] * 100
    del on_means, rest_means
    diff_map = np.where(valid_vox, diffs.mean(axis=-1), 0)
    sd = diffs.std(axis=-1, ddof=1)
    t_map = np.where(valid_vox & (sd > 0), diff_map / (sd / np.sqrt(len(onsets)) + (sd == 0)), 0)
    del diffs
    if POSITIVE_ONLY:
        diff_map = np.clip(diff_map, 0, None)
        t_map = np.clip(t_map, 0, None)

    native_maps = atlas_dir / "native_maps"
    native_maps.mkdir(exist_ok=True)
    for name, data in (("stim_diff_psc", diff_map), ("stim_tstat", t_map)):
        save_map(data, native_maps / f"{name}.nii.gz")

    # Native -> atlas: inverse of the functional -> atlas matrix, resampled once onto the registration template grid
    atlas_space = atlas_dir / "atlas_space"
    atlas_space.mkdir(exist_ok=True)
    atlas_grid = prepare_registration_template(atlas_template, atlas_mask_text, species_cfg["reg_voxel_mm"], atlas_dir)
    native_to_atlas = atlas_dir / "native_to_atlas.aff12.1D"
    res = run_afni(["cat_matvec", "-ONELINE", composite, "-I"], capture=True)
    native_to_atlas.write_text(res.stdout.strip() + "\n")

    for name, source, interp in (("stim_diff_psc", native_maps / "stim_diff_psc.nii.gz", "linear"),
                                 ("stim_tstat", native_maps / "stim_tstat.nii.gz", "linear"),
                                 ("mean_func", func_reference, "linear"),
                                 ("brain_mask", mask_file, "NN")):
        run_afni(["3dAllineate", "-source", source, "-base", atlas_grid, "-master", atlas_grid, "-1Dmatrix_apply", native_to_atlas,
                  "-final", interp, "-prefix", atlas_space / f"{name}_atlasspace.nii.gz", "-overwrite"])
    print_statement(f"[OK] Maps in atlas space: {atlas_space}", bcolors.OKGREEN)

    # Check the alignment of the warped maps on the template
    if OPEN_FSLEYES:
        subprocess.run(["fsleyes", str(atlas_grid), str(atlas_space / "stim_tstat_atlasspace.nii.gz"), "-cm", "hot", "-dr", "2", "6"])
else:
    print_statement("⏭️ Stimulation response not selected (tick it in the first cell)", bcolors.NOTIFICATION)

# Part 2: Post-stimulation Analysis (LTP)

Looks at the slow signal change after the stimulation instead of the acute response.

## Baseline and Signal Window

Start indices are volumes (0-based) of the stitched series, see the scan ranges printed above; the window duration is entered in **seconds** and converted with the volume TR (the number of volumes is shown next to it). Both windows have the same length (default 300 s). The defaults follow the last scan: baseline 50 s after its start (the signal is still settling right after a scan start), signal window starting at 3/4 of it. SORDINO: if the scan ends with a long trigger block (stimulation blocks and the post-stimulation phase in one scan), the baseline window starts at the beginning of that phase and the signal window ends at the end of the scan; the sordino scans have no start-up transient (10 s of dummy scans before every scan), so the offset can be shortened. Check the whole-brain time course below before trusting the windows: a sudden step in the signal (e.g. 466 scan 36 around volume 325 of the scan) is not a stimulation effect and must stay out of both windows.

Percent signal change is relative to the mean of the baseline window. To compare the two windows, each window is cut into non-overlapping bins (`BIN_S` seconds); a Welch t-test then compares the signal bins with the baseline bins. Testing bin means instead of single volumes keeps the strong temporal autocorrelation of the fMRI signal from inflating the statistics.

In [ ]:
if RUN_POSTSTIM:
    BASELINE_OFFSET_S = 50  # seconds after the start of the last scan; the signal is still settling right after a scan start
    WINDOW_S = 300  # default length of the baseline and the signal window
    post_scan, post_start, post_end = scan_ranges[-1]
    n_post = post_end - post_start + 1

    default_base_start = post_start + int(round(BASELINE_OFFSET_S / tr))
    default_sig_start = post_start + int(0.75 * n_post)
    default_win_s = WINDOW_S

    # SORDINO: one scan with the stimulation blocks followed by a much longer trigger block (the post-stimulation phase): the windows
    # are the start and the end of that phase
    scan_blocks = session.get("scan_info", {}).get(str(func_scan_numbers[-1]), {})
    vpb, onsets_in_scan = scan_blocks.get("volumes_per_block") or [], scan_blocks.get("block_onsets") or []
    if len(vpb) > 2 and vpb[-1] >= 2 * min(vpb[:-1]):
        phase_start = post_start + onsets_in_scan[-1] - session.get("drop_vols", 0)
        n_phase = post_end + 1 - phase_start
        default_win_vols = min(int(round(WINDOW_S / tr)), n_phase // 2)
        default_win_s = default_win_vols * tr
        default_base_start = phase_start
        default_sig_start = post_end + 1 - default_win_vols
        print_statement(f"Scan {func_scan_numbers[-1]} ends with a post-stimulation block of {n_phase} volumes ({n_phase * tr / 60:.0f} min, from volume {phase_start}): "
                        f"baseline window at its start, signal window at its end.", bcolors.NOTIFICATION)


    def window_volumes(seconds):
        """Window duration in seconds -> number of volumes (at least 1)."""
        return max(1, int(round(float(seconds) / tr)))


    input_layout = widgets.Layout(width="280px", flex="0 0 auto")
    label_style = {'description_width': '180px'}
    base_start_idx = widgets.IntText(value=default_base_start, description="Baseline Start Index:", layout=input_layout, style=label_style)
    sig_start_idx = widgets.IntText(value=default_sig_start, description="Signal Start Index:", layout=input_layout, style=label_style)
    win_seconds_entered = widgets.FloatText(value=round(default_win_s, 2), description="Window Duration (s):", layout=input_layout, style=label_style)
    win_info = widgets.HTML()


    def update_win_info(change=None):
        win_info.value = f"= {window_volumes(win_seconds_entered.value)} volumes (TR {tr:.3f} s)"


    win_seconds_entered.observe(update_win_info, names="value")
    update_win_info()
    display(widgets.HBox([base_start_idx, sig_start_idx, win_seconds_entered, win_info],
        layout=widgets.Layout(width="100%", flex_flow="row wrap", justify_content="space-between", align_items="center", padding="12px", border="3px solid #444", border_radius="10px", background_color="#1e1e1e")))
else:
    print_statement("⏭️ Post-stimulation analysis not selected (tick it in the first cell)", bcolors.NOTIFICATION)

In [ ]:
if RUN_POSTSTIM:
    BIN_S = 30  # length of the bins that are tested against each other

    base_start, sig_start, win = int(base_start_idx.value), int(sig_start_idx.value), window_volumes(win_seconds_entered.value)
    base_end, sig_end = base_start + win, sig_start + win
    if min(base_start, sig_start) < 0 or max(base_end, sig_end) > n_total:
        raise ValueError(f"Windows must lie within the stitched series (0-{n_total - 1}).")
    if base_start < sig_end and sig_start < base_end:
        raise ValueError("Baseline and signal window overlap.")

    bin_vols = max(1, int(round(BIN_S / tr)))
    n_bins = win // bin_vols
    if n_bins < 3:
        raise ValueError(f"Window of {win} volumes holds only {n_bins} bins of {BIN_S} s; increase the window or decrease BIN_S.")


    def window_bins(x, start):
        """Mean of every non-overlapping bin of the window starting at `start` (last axis is time)."""
        seg = x[..., start:start + n_bins * bin_vols]
        return seg.reshape(*seg.shape[:-1], n_bins, bin_vols).mean(axis=-1)


    def scan_boundaries():
        return [start for _, start, _ in scan_ranges[1:]]


    def mark_windows(ax, to_time=lambda v: v * tr / 60):
        ax.axvspan(to_time(base_start), to_time(base_end), color="green", alpha=0.3, label="Baseline")
        ax.axvspan(to_time(sig_start), to_time(sig_end), color="blue", alpha=0.3, label="Signal")
        for b in scan_boundaries():
            ax.axvline(to_time(b), color="k", ls="--", lw=0.8)


    print_statement(f"Baseline window: volumes {base_start}-{base_end - 1}; signal window: volumes {sig_start}-{sig_end - 1} ({win} volumes = {win * tr:.0f} s); "
                    f"{n_bins} bins of {bin_vols} volumes ({bin_vols * tr:.0f} s) each", bcolors.OKGREEN)

    # Whole-brain mean as a check of the window choice (dashed lines = scan boundaries)
    brain_base = window_bins(brain_ts, base_start).mean()
    fig, ax = plt.subplots(figsize=(14, 3.5))
    ax.plot(np.arange(n_total) * tr / 60, (brain_ts - brain_base) / brain_base * 100, lw=0.6)
    mark_windows(ax)
    ax.set_xlabel("Time in stitched series (min)")
    ax.set_ylabel("Brain mean signal change (%)")
    ax.legend(loc="upper right")
    plt.tight_layout()
    plt.show()
else:
    print_statement("⏭️ Post-stimulation analysis not selected (tick it in the first cell)", bcolors.NOTIFICATION)

## Signal Change per Brain Region

The mean time course of every atlas region (inside the brain mask) is extracted, and the signal window is compared with the baseline window (Welch t-test on the bins), with Benjamini-Hochberg FDR correction across regions. This replaces manually drawn ROIs. Outputs go to `roi_atlas_<species>/poststim/`:
- `poststim_roi_response.csv`: every region with baseline-normalised signal change, t, p and FDR-corrected q; `poststim_roi_responders.csv` lists only the regions with a significant increase.
- `poststim_roi_psc_timeseries_all.csv`: percent signal change time course of every region.
- `poststim_roi_response_map.nii.gz`: signal change of the responding regions on the functional grid.
- Figures (in `<subject>/figures/results/<functional folder>_<analysis>/poststim/`): `poststim_roi_response_top_increases.svg` / `..._top_decreases.svg`: time courses of the `N_TOP` regions with the strongest signal increase and the strongest decrease (ranked by t, or by the size of the signal change with `RANK_BY`); the same regions are printed as tables.
- `plots/` (in the same figure folder): one figure per region (time course with both windows and the bin values); the responders are also in `plots/responders/`.

In [ ]:
if RUN_POSTSTIM:
    from scipy import stats
    from scipy.ndimage import uniform_filter1d
    from statsmodels.stats.multitest import multipletests

    MIN_VOXELS = 5
    FDR_ALPHA = 0.05
    N_TOP = 10  # regions shown per direction: the N_TOP strongest signal increases and the N_TOP strongest decreases
    RANK_BY = "t"  # "t": by t-statistic (significance); "signal_psc": by size of the signal change in % (favours small, noisy regions)
    PLOT_SMOOTH_S = 60  # smoothing of the plotted time courses
    SAVE_ALL_REGION_PLOTS = True  # one figure per region in plots/ (responders always saved)

    post_dir = atlas_dir / "poststim"
    post_dir.mkdir(exist_ok=True)
    smooth_vols = max(1, int(round(PLOT_SMOOTH_S / tr)))
    t_min = np.arange(n_total) * tr / 60
    post_fig_dir = fig_dirs["results"] / "poststim"  # figures; tables stay in post_dir
    plot_dir = post_fig_dir / "plots"
    (plot_dir / "responders").mkdir(parents=True, exist_ok=True)

    rows, region_psc, region_bins = [], {}, {}
    for lid in np.unique(labels[labels > 0]):
        sel = labels == lid
        n_vox = int(sel.sum())
        if n_vox < MIN_VOXELS:
            continue
        ts = func_4d[sel].mean(axis=0)
        b_bins, s_bins = window_bins(ts, base_start), window_bins(ts, sig_start)
        base = b_bins.mean()
        if base <= 0:
            continue
        t_stat, p_val = stats.ttest_ind(s_bins, b_bins, equal_var=False)
        name = label_names.get(int(lid), f"label_{lid}")
        region_psc[lid] = (ts - base) / base * 100
        region_bins[lid] = ((b_bins - base) / base * 100, (s_bins - base) / base * 100)
        rows.append({"label_id": int(lid), "name": name, "n_voxels": n_vox,
                     "baseline": base, "signal_psc": (s_bins.mean() - base) / base * 100, "t": t_stat, "p": p_val})

    results = pd.DataFrame(rows)
    valid = results["p"].notna()
    results.loc[valid, "q_fdr"] = multipletests(results.loc[valid, "p"], alpha=FDR_ALPHA, method="fdr_bh")[1]
    results = results.sort_values("p").reset_index(drop=True)
    results["responder"] = (results["q_fdr"] < FDR_ALPHA) & (results["signal_psc"] > 0)

    results.to_csv(post_dir / "poststim_roi_response.csv", index=False)
    results[results["responder"]].to_csv(post_dir / "poststim_roi_responders.csv", index=False)
    pd.DataFrame({results.loc[results.label_id == lid, "name"].iloc[0]: psc for lid, psc in region_psc.items()}).to_csv(
        post_dir / "poststim_roi_psc_timeseries_all.csv", index_label="volume")

    print_statement(f"{len(results)} regions analysed, {int(results['responder'].sum())} with a significant signal increase (FDR q < {FDR_ALPHA})", bcolors.OKGREEN)
    print(results.head(20).to_string(index=False))

    # Map of the signal change in regions that respond
    response_map = np.zeros(labels.shape, dtype=np.float32)
    for r in results[results["responder"]].itertuples():
        response_map[labels == r.label_id] = r.signal_psc
    save_map(response_map, post_dir / "poststim_roi_response_map.nii.gz")


    def plot_time_course(ax, r, title_size=9):
        psc = region_psc[r.label_id]
        ax.plot(t_min, psc, lw=0.4, alpha=0.4)
        ax.plot(t_min, uniform_filter1d(psc, smooth_vols), lw=1.2, color="C0")
        mark_windows(ax)
        ax.set_xlabel("Time in stitched series (min)")
        ax.set_ylabel("Signal change (%)")
        ax.set_title(f"{r.name}\nΔ={r.signal_psc:.2f}%, q={r.q_fdr:.3g}", fontsize=title_size)


    # Strongest regions: the N_TOP strongest signal increases and the N_TOP strongest decreases
    def top_regions(sign):
        """The N_TOP regions with the strongest signal increase (sign=1) or decrease (sign=-1), ranked by RANK_BY."""
        column = "t" if RANK_BY == "t" else "signal_psc"
        candidates = results[(results["signal_psc"] * sign > 0) & results[column].notna()]
        return candidates.loc[(candidates[column] * sign).sort_values(ascending=False).index[:N_TOP]]


    def plot_top_regions(top, title, out_file):
        n_rows = max(1, int(np.ceil(len(top) / 3)))
        fig, axes = plt.subplots(n_rows, 3, figsize=(18, 3.5 * n_rows), squeeze=False)
        for ax, r in zip(axes.flat, top.itertuples()):
            plot_time_course(ax, r)
        for ax in list(axes.flat)[len(top):]:
            ax.axis("off")
        fig.suptitle(title)
        fig.tight_layout(rect=(0, 0, 1, 0.97))
        fig.savefig(out_file, dpi=300)
        plt.show()


    top_columns = ["name", "n_voxels", "baseline", "signal_psc", "t", "p", "q_fdr"]
    for sign, label in ((1, "increase"), (-1, "decrease")):
        top = top_regions(sign)
        print_statement(f"{len(top)} strongest signal {label}s (ranked by {RANK_BY}):", bcolors.NOTIFICATION)
        print(top[top_columns].to_string(index=False))
        plot_top_regions(top, f"Strongest signal {label}s (ranked by {RANK_BY})", post_fig_dir / f"poststim_roi_response_top_{label}s.svg")


    def save_region_plot(r):
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4), gridspec_kw={"width_ratios": [3, 1]})
        plot_time_course(ax1, r, title_size=11)
        ax1.set_title(f"{r.name} ({r.n_voxels} voxels)")
        b, s = region_bins[r.label_id]
        ax2.boxplot([b, s], showfliers=False)
        for i, vals in enumerate((b, s), start=1):
            ax2.scatter(np.full(len(vals), i) + np.random.default_rng(0).uniform(-0.1, 0.1, len(vals)), vals, s=10, color="k", zorder=3)
        ax2.set_xticks([1, 2], ["Baseline", "Signal"])
        ax2.set_ylabel(f"Signal change (%), {BIN_S} s bins")
        ax2.set_title(f"t={r.t:.2f}, p={r.p:.3g}, q={r.q_fdr:.3g}")
        fig.tight_layout()
        fname = f"poststim_{r.label_id}_{safe_name(r.name)}.svg"
        if SAVE_ALL_REGION_PLOTS:
            fig.savefig(plot_dir / fname, dpi=300)
        if r.responder:
            fig.savefig(plot_dir / "responders" / fname, dpi=300)
        plt.close(fig)


    for r in results.itertuples():
        if SAVE_ALL_REGION_PLOTS or r.responder:
            save_region_plot(r)
    print_statement(f"Figures saved in {plot_dir} ({int(results['responder'].sum())} responders in {plot_dir / 'responders'})", bcolors.OKGREEN)
else:
    print_statement("⏭️ Post-stimulation analysis not selected (tick it in the first cell)", bcolors.NOTIFICATION)

## Voxelwise Signal Change Maps in Atlas Space (for group analysis)

The region analysis above stays in native space, so the functional time series are never interpolated. For voxelwise group statistics, the maps are computed in native space and then moved to atlas space with the inverse of the functional → atlas matrix (struct2func and struct2atlas), in a single resampling step:
- `poststim_diff_psc`: mean signal-window minus baseline-window signal, in % of the baseline, per voxel (the signal change map of `mol_fMRI_v5.4.ipynb`).
- `poststim_tstat`: Welch t of the signal bins vs. the baseline bins per voxel.
- `mean_func` and `brain_mask`, for overlays and for restricting the group statistics to the brain.

The maps contain signal increases only (negative values are set to 0); set `POSITIVE_ONLY = False` in the code cell to keep decreases as well. The bin images are smoothed with a Gaussian kernel (`SMOOTH_FWHM_MM`, within the brain mask) before the maps are computed. Set it to 0 to switch smoothing off.

Outputs go to `roi_atlas_<species>/poststim/native_maps/` and `.../atlas_space/` (on the registration template grid, which is the same for every animal of the species). Region-wise group comparisons can use `poststim_roi_response.csv` directly, since the atlas labels already define the same regions in every animal.

In [ ]:
if RUN_POSTSTIM:
    from scipy.ndimage import gaussian_filter

    POSITIVE_ONLY = True  # only signal increases in the maps: negative values are set to 0 (False keeps both directions)
    SMOOTH_FWHM_MM = 0.3  # as in mol_fMRI_v5.4 (rat); 0 = no smoothing, probably smaller for mouse

    zooms = np.array(labels_img.header.get_zooms()[:3], dtype=float)
    sigma_vox = SMOOTH_FWHM_MM / 2.3548 / zooms
    brain_f = brain.astype(np.float32)
    brain_weight = gaussian_filter(brain_f, sigma_vox) if SMOOTH_FWHM_MM > 0 else brain_f


    def smooth_in_brain(img):
        """Gaussian smoothing that does not pull in signal from outside the brain mask."""
        if SMOOTH_FWHM_MM <= 0:
            return img * brain_f
        return np.where(brain_weight > 0, gaussian_filter(img * brain_f, sigma_vox) / np.where(brain_weight > 0, brain_weight, 1), 0)


    def smoothed_bins(start):
        bins = window_bins(func_4d, start)
        return np.stack([smooth_in_brain(bins[..., k]) for k in range(bins.shape[-1])], axis=-1)


    base_bins, sig_bins = smoothed_bins(base_start), smoothed_bins(sig_start)
    base_img = base_bins.mean(axis=-1)
    valid_vox = brain & (base_img > 0)
    diff_map = np.where(valid_vox, (sig_bins.mean(axis=-1) - base_img) / np.where(valid_vox, base_img, 1) * 100, 0)
    with np.errstate(divide="ignore", invalid="ignore"):
        t_map = stats.ttest_ind(sig_bins, base_bins, axis=-1, equal_var=False).statistic
    t_map = np.where(valid_vox, np.nan_to_num(t_map), 0)
    if POSITIVE_ONLY:
        diff_map = np.clip(diff_map, 0, None)
        t_map = np.clip(t_map, 0, None)

    native_maps = post_dir / "native_maps"
    native_maps.mkdir(exist_ok=True)
    for name, data in (("poststim_diff_psc", diff_map), ("poststim_tstat", t_map)):
        save_map(data, native_maps / f"{name}.nii.gz")

    # Native -> atlas: inverse of the functional -> atlas matrix, resampled once onto the registration template grid
    atlas_space = post_dir / "atlas_space"
    atlas_space.mkdir(exist_ok=True)
    atlas_grid = prepare_registration_template(atlas_template, atlas_mask_text, species_cfg["reg_voxel_mm"], atlas_dir)
    native_to_atlas = atlas_dir / "native_to_atlas.aff12.1D"
    res = run_afni(["cat_matvec", "-ONELINE", composite, "-I"], capture=True)
    native_to_atlas.write_text(res.stdout.strip() + "\n")

    for name, source, interp in (("poststim_diff_psc", native_maps / "poststim_diff_psc.nii.gz", "linear"),
                                 ("poststim_tstat", native_maps / "poststim_tstat.nii.gz", "linear"),
                                 ("mean_func", func_reference, "linear"),
                                 ("brain_mask", mask_file, "NN")):
        run_afni(["3dAllineate", "-source", source, "-base", atlas_grid, "-master", atlas_grid, "-1Dmatrix_apply", native_to_atlas,
                  "-final", interp, "-prefix", atlas_space / f"{name}_atlasspace.nii.gz", "-overwrite"])
    print_statement(f"[OK] Maps in atlas space: {atlas_space}", bcolors.OKGREEN)

    # Check the alignment of the warped maps on the template
    if OPEN_FSLEYES:
        subprocess.run(["fsleyes", str(atlas_grid), str(atlas_space / "poststim_tstat_atlasspace.nii.gz"), "-cm", "hot", "-dr", "2", "6"])
else:
    print_statement("⏭️ Post-stimulation analysis not selected (tick it in the first cell)", bcolors.NOTIFICATION)